# Project main notebook
You will find here labelled functionalities

## 1. Dataset 
### 1.1 Loading

In [ ]:
from src.utils.params import Params, integrate_global_parameters
from src.dataset.dataset import ADE20KDataset, make_train_transform

global_parameters = Params()
global_parameters = integrate_global_parameters(global_parameters)
dataset = ADE20KDataset(global_parameters.get("train_image_folder"),
                        global_parameters.get("train_annotation_folder"),
                        transform=make_train_transform(
                            mean = global_parameters.get("mean", (0.485, 0.456, 0.406)),
                            std = global_parameters.get("std", (0.229, 0.224, 0.225))
                            )
                        )

### 1.2 Generating sample image-annotation pairs

In [ ]:
from src.dataset.dataset import get_random_subset
from src.utils.drawing import color_np_mask
from PIL import Image
import numpy as np
clean_dataset = ADE20KDataset(global_parameters.get("train_image_folder"),
                        global_parameters.get("train_annotation_folder")
                        )
subset_dataset = get_random_subset(clean_dataset, 10, seed=42)
for i in range(10):
    image = np.array(subset_dataset[i]["image"]) 
    mask = np.array(subset_dataset[i]["annotation"])
    colored_mask = color_np_mask(mask)
    joint_result = np.concatenate((image, colored_mask), axis=1)
    joint_result = Image.fromarray(joint_result)
    joint_result.save(f"output/dataset_examples/image_{i}.png")

### 1.3 Calculating dataset mean and std

In [ ]:
from src.preprocessing.dataset_preprocessing import preprocess_ade20k
preprocess_ade20k(global_parameters)

## 2. Model preprocessing (optional)

### 2.1 Hyperparameter estimation

In [ ]:
from src.preprocessing.hyperparameter_estimation import estimate_hyperparameters
from src.models.U_net import UNet
param_path = 'src/config/uNet_params.json'
model_parameters = Params(param_path)
model = UNet(model_parameters) # UNet used only as an example, you can use any other model
estimate_hyperparameters(model=model, model_params_path=param_path,
                         dataset=dataset, n_trials=20, sample_size=500, epochs=15)
# it will print trials, results and best hyperparameters, as well as save them to param_path

### 2.2 Overfitting on small subset of data

In [ ]:
from src.preprocessing.verify_model import verify_model_memorization
# Important: you can also set parameters in ./src/config/memorization_params.json
verify_model_memorization(model = model, dataset=dataset, sample_size=8, params=model_parameters, epochs=150)

## 3. Models validation

### 3.1 Model loading

In [ ]:
# Load models for comparison
import torch
from src.models.U_net import UNet
from src.models.mask2former import mask2former
from src.models.convnext import Convnext

# Initialize device
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# Load UNet model (trained)
unet_model = UNet(in_channels=3, base_channels=64, num_classes=151)
unet_model.load_state_dict(torch.load('models/best_unet.pt', map_location=device))
unet_model.to(device)
unet_model.eval()

print("Loaded UNet model")

# Load Mask2Former (pre-trained on ADE20K)
# Note: This will download ~1GB on first run
mask2former_model = mask2former(num_classes=150)
mask2former_model.to(device)
mask2former_model.eval()

print("Loaded Mask2Former model")

# Load ConvNeXt (pre-trained on ADE20K)
# Note: This will download ~400MB on first run
convnext_model = Convnext(model_id="openmmlab/upernet-convnext-base", device=device)
convnext_model.eval()
print("Loaded ConvNeXt model")

# List of models to compare
models = [unet_model, mask2former_model, convnext_model]

print(f"\n{len(models)} models loaded and ready for comparison")

### 3.2 Compare model metrics and losses

In [ ]:
from src.dataset.dataset import get_random_subset
from src.validation.compare_models import compare_models, compare_models_visually
from src.dataset.dataset import make_val_transform
# Quantitative comparison on validation subset
from src.utils.params import Params

# Create a small validation subset for quick comparison
validation_dataset = ADE20KDataset(global_parameters.get("validation_image_folder"),
                        global_parameters.get("validation_annotation_folder"),
                        transform=make_val_transform(
                            mean = global_parameters.get("mean", (0.485, 0.456, 0.406)),
                            std = global_parameters.get("std", (0.229, 0.224, 0.225))
                            )
                        )
val_subset = validation_dataset
# if you wish to use only a subset for faster testing, uncomment the next line
# val_subset = get_random_subset(validation_dataset, 100)

# Set up parameters for validation
comparison_params = Params()
comparison_params.set("device", device)
comparison_params.set("batch_size", 2)
comparison_params.set("num_workers", 0)
comparison_params.set("num_classes", 151)
comparison_params.set("ignore_index", 0)
comparison_params.set("validation_loss_functions", {
    "CrossEntropyLoss": "CrossEntropyLoss",
    "DiceLoss": "DiceLoss"
})
comparison_params.set("validation_loss_params", {"ignore_index": 0})

# Run comparison
results = compare_models(
    models=models[:1],
    dataset=val_subset,
    params=comparison_params,
    output_directory="./output/model_comparison/"
)

# Display results
print("QUANTITATIVE COMPARISON RESULTS")
for res in results:
    print(f"\n{res['model_name']}:")
    for metric, value in res['results'].items():
        print(f"  {metric:20s}: {value:.4f}")

print("\n Results saved to: ./output/model_comparison/comparison_results.csv")

### 3.3 Visual comparison

In [ ]:
# Visual comparison on a few samples
print("Creating visual comparison...")

# Set parameters for visual comparison
visual_params = Params()
visual_params.set("input_height", 512)
visual_params.set("input_width", 512)

# Compare visually on 10 random samples (put amount of samples you want to see in sample_number)
compare_models_visually(
    models=models,
    dataset=validation_dataset,
    params=visual_params,
    output_directory="./output/visual_comparison/",
    sample_number=10
)

print("Visual comparisons saved to: ./output/visual_comparison/")

## 4: Image inpainting

In [ ]:
from lama.simple_lama import SimpleLama
from src.models.mask2former import mask2former
from src.utils.drawing import enlarge_mask
from src.predict import predict_image
import os
import cv2
import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

target_mask = 13 # here choose any object class you want to inpaint (13 is for "person")
# For list of classes, see: ./data/ADEChallenge/ADE20K_objectInfo150.txt
# all images with target_mask
images_with_target_mask = []
for i in range(len(dataset)):
    if i % 1000 == 0:
        print(i)
    annotation = np.array(dataset[i]["annotation"])
    if target_mask in annotation:
        #print("succes")
        images_with_target_mask.append(i)

simple_lama : SimpleLama = SimpleLama(model_path="models/big-lama.pt")
mask2former_model = mask2former()

for i in range(10):
    image = np.array(dataset[images_with_target_mask[i]]["image"]) 
    annotation = np.array(dataset[images_with_target_mask[i]]["annotation"])
    image = image.transpose(1, 2, 0)
    mask = predict_image(mask2former_model, image, save_image=False) 
    print(mask[:10, :10])
    compare_mask = mask == target_mask
    dilated_mask = enlarge_mask(compare_mask, 8)
    dilated_mask = (dilated_mask * 255).astype(np.uint8)
    print(image.shape, dilated_mask.shape)
    
    # Ensure mask is single channel (grayscale)
    mask_image = Image.fromarray(dilated_mask).convert('L')
    parent_dir = "output/lama_inpainted"
    os.makedirs(parent_dir, exist_ok=True)
    image = (image * 255).astype(np.uint8)
    # run inpainting
    result = simple_lama(image, mask_image)

    # save result
    result = cv2.resize(np.array(result), (image.shape[1], image.shape[0]))
    joint_result = np.concatenate((image, np.array(result)), axis=1)
    joint_result = Image.fromarray(joint_result)
    joint_result.save(f"output/lama_inpainted/image_{i}.png")
    result

    # create plot where on left is original image with mask and on right is inpainted image
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))  # 1 row, 2 columns
    axes[0].axis('off')
    axes[1].axis('off')

    axes[0].imshow(image)
    axes[1].imshow(result)
    plt.show()